# 🏥 Entrenamiento RoBERTa-BSC para Clasificación CIE-10

Sistema de **Extreme Multi-label Classification (XMC)** para códigos CIE-10 usando el modelo **RoBERTa-BSC** del Plan TL con el dataset **CODIESP**.

## Características:
- 🤖 **Modelo:** `PlanTL-GOB-ES/roberta-base-bne-medical` (RoBERTa BSC especializado en español médico)
- 📚 **Dataset:** CODIESP (corpus clínico español con códigos CIE-10)
- 🎯 **Tarea:** Clasificación multilabel jerárquica
- 💡 **Explicabilidad:** Sistema RAG para justificar predicciones
- 🖥️ **Hardware:** Optimizado para CPU (con opción GPU)

---

In [ ]:
# Verificar instalación de paquetes
import sys
!{sys.executable} -m pip install --quiet transformers>=4.30.0 datasets>=2.14.0 torch>=2.0.0 \
    scikit-learn>=1.3.0 pandas>=2.0.0 numpy>=1.24.0 tqdm>=4.65.0 \
    faiss-cpu>=1.7.4 sentence-transformers>=2.2.0

print("✓ Dependencias instaladas correctamente")

In [ ]:
# Imports necesarios
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AdamW, get_linear_schedule_with_warmup
from datasets import load_dataset
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from tqdm.auto import tqdm
import os
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuración del dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Dispositivo: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
else:
    print("   ⚠️  Entrenando en CPU (será más lento)")

# Configuración de rutas
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT.parent / "csv_import_scripts" / "codiesp_csvs"
SNAPSHOTS_DIR = PROJECT_ROOT / "snapshots"
MODEL_DIR = SNAPSHOTS_DIR / "roberta_bsc_cie10"
RAG_DIR = SNAPSHOTS_DIR / "rag_index"

# Crear directorios
SNAPSHOTS_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
RAG_DIR.mkdir(exist_ok=True)

print(f"\n📂 Rutas configuradas:")
print(f"   Dataset: {DATA_DIR}")
print(f"   Modelo: {MODEL_DIR}")
print(f"   RAG: {RAG_DIR}")

In [ ]:
class Config:
    """Configuración del modelo y entrenamiento"""

    # Modelo
    MODEL_NAME = 'PlanTL-GOB-ES/roberta-base-bne-medical'  # RoBERTa-BSC médico
    MAX_LENGTH = 512  # Longitud máxima de secuencia

    # Entrenamiento (optimizado para CPU)
    if device.type == 'cuda':
        TRAIN_BATCH_SIZE = 16
        EVAL_BATCH_SIZE = 32
        ACCUMULATION_STEPS = 2
    else:
        TRAIN_BATCH_SIZE = 4   # Batch pequeño para CPU
        EVAL_BATCH_SIZE = 8
        ACCUMULATION_STEPS = 4  # Acumular gradientes para simular batch mayor

    EPOCHS = 10  # Para pruebas rápidas en CPU (aumentar a 50+ en GPU)
    LEARNING_RATE = 2e-5
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01
    MAX_GRAD_NORM = 1.0

    # Dataset
    TRAIN_PATH = DATA_DIR / 'codiesp_D_source_train.csv'
    VAL_PATH = DATA_DIR / 'codiesp_D_source_validation.csv'
    TEST_PATH = DATA_DIR / 'codiesp_D_source_test.csv'

    # Guardado de modelo
    SAVE_STEPS = 500
    EVAL_STEPS = 500
    SAVE_TOTAL_LIMIT = 3

    # RAG
    RAG_TOP_K = 5  # Recuperar top-5 fragmentos más relevantes
    RAG_CHUNK_SIZE = 200  # Tamaño de chunks del manual CIE-10

config = Config()

print("⚙️  Configuración:")
print(f"   Modelo: {config.MODEL_NAME}")
print(f"   Batch size (train): {config.TRAIN_BATCH_SIZE}")
print(f"   Batch size (eval): {config.EVAL_BATCH_SIZE}")
print(f"   Épocas: {config.EPOCHS}")
print(f"   Learning rate: {config.LEARNING_RATE}")
print(f"   Acumulación de gradientes: {config.ACCUMULATION_STEPS}")
print(f"   Max length: {config.MAX_LENGTH}")

In [ ]:
# Cargar datasets
print("📥 Cargando datasets CODIESP...")

train_df = pd.read_csv(config.TRAIN_PATH)
val_df = pd.read_csv(config.VAL_PATH)
test_df = pd.read_csv(config.TEST_PATH)

print(f"\n📊 Estadísticas del dataset:")
print(f"   Train: {len(train_df)} documentos")
print(f"   Val:   {len(val_df)} documentos")
print(f"   Test:  {len(test_df)} documentos")

# Explorar estructura
print(f"\n🔍 Columnas: {train_df.columns.tolist()}")
print(f"\n📄 Ejemplo de documento:")
print(f"   Texto: {train_df.iloc[0]['text'][:200]}...")
print(f"   Labels: {train_df.iloc[0]['labels']}")

In [ ]:
# Preprocesar labels (convertir string a lista)
def parse_labels(label_str):
    """Convertir string de labels a lista"""
    if pd.isna(label_str):
        return []
    try:
        return eval(label_str) if isinstance(label_str, str) else label_str
    except:
        return []

train_df['labels'] = train_df['labels'].apply(parse_labels)
val_df['labels'] = val_df['labels'].apply(parse_labels)
test_df['labels'] = test_df['labels'].apply(parse_labels)

# Construir vocabulario de códigos CIE-10
all_codes = set()
for labels in train_df['labels']:
    all_codes.update(labels)
for labels in val_df['labels']:
    all_codes.update(labels)

all_codes = sorted(list(all_codes))
code2id = {code: idx for idx, code in enumerate(all_codes)}
id2code = {idx: code for code, idx in code2id.items()}

print(f"\n🏷️  Total de códigos CIE-10 únicos: {len(all_codes)}")
print(f"   Ejemplos: {all_codes[:10]}")

# MultiLabelBinarizer para convertir a formato binario
mlb = MultiLabelBinarizer(classes=all_codes)
mlb.fit([all_codes])

# Guardar mapeo de códigos
with open(MODEL_DIR / 'code2id.json', 'w') as f:
    json.dump(code2id, f, indent=2)

print(f"✓ Mapeo de códigos guardado en {MODEL_DIR / 'code2id.json'}")

In [ ]:
class CIE10Dataset(Dataset):
    """Dataset para clasificación multilabel CIE-10"""

    def __init__(self, dataframe, tokenizer, mlb, max_length=512):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.mlb = mlb
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        text = str(row['text'])
        labels = row['labels']

        # Tokenizar texto
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        # Convertir labels a formato binario
        label_binary = self.mlb.transform([labels])[0].astype(np.float32)

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label_binary, dtype=torch.float)
        }

print("✓ Dataset personalizado definido")

In [ ]:
# Cargar tokenizer de RoBERTa-BSC
print(f"📥 Cargando tokenizer: {config.MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)

# Crear datasets
train_dataset = CIE10Dataset(train_df, tokenizer, mlb, config.MAX_LENGTH)
val_dataset = CIE10Dataset(val_df, tokenizer, mlb, config.MAX_LENGTH)

# Crear dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config.TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=0  # 0 para CPU, aumentar en GPU
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(f"✓ Tokenizer y dataloaders listos")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")

In [ ]:
class RoBERTaCIE10Classifier(nn.Module):
    """Modelo RoBERTa-BSC para clasificación multilabel CIE-10"""

    def __init__(self, model_name, num_labels):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.roberta.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Usar el [CLS] token (primer token)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        return logits

# Inicializar modelo
num_labels = len(all_codes)
print(f"🤖 Inicializando modelo: {config.MODEL_NAME}")
print(f"   Número de etiquetas: {num_labels}")

model = RoBERTaCIE10Classifier(config.MODEL_NAME, num_labels)
model = model.to(device)

# Contar parámetros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 Parámetros del modelo:")
print(f"   Total: {total_params:,}")
print(f"   Entrenables: {trainable_params:,}")

In [ ]:
# Configurar optimizador
optimizer = AdamW(
    model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

# Calcular pasos totales
total_steps = len(train_loader) * config.EPOCHS // config.ACCUMULATION_STEPS
warmup_steps = int(total_steps * config.WARMUP_RATIO)

# Scheduler con warmup
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# Función de pérdida: Binary Cross Entropy with Logits
criterion = nn.BCEWithLogitsLoss()

print(f"⚙️  Entrenamiento configurado:")
print(f"   Total steps: {total_steps}")
print(f"   Warmup steps: {warmup_steps}")
print(f"   Optimizer: AdamW")
print(f"   Loss: BCEWithLogitsLoss")

In [ ]:
# Funciones auxiliares para evaluación
def compute_metrics(predictions, labels, threshold=0.5):
    """Calcular métricas para clasificación multilabel"""
    preds_binary = (predictions >= threshold).astype(int)

    f1_micro = f1_score(labels, preds_binary, average='micro', zero_division=0)
    f1_macro = f1_score(labels, preds_binary, average='macro', zero_division=0)
    precision_micro = precision_score(labels, preds_binary, average='micro', zero_division=0)
    recall_micro = recall_score(labels, preds_binary, average='micro', zero_division=0)

    return {
        'f1_micro': f1_micro,
        'f1_macro': f1_macro,
        'precision_micro': precision_micro,
        'recall_micro': recall_micro
    }

def evaluate(model, dataloader, criterion, device):
    """Evaluar modelo en conjunto de validación"""
    model.eval()
    total_loss = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluando"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            total_loss += loss.item()

            # Aplicar sigmoid para obtener probabilidades
            predictions = torch.sigmoid(logits).cpu().numpy()
            all_predictions.append(predictions)
            all_labels.append(labels.cpu().numpy())

    # Concatenar todas las predicciones
    all_predictions = np.vstack(all_predictions)
    all_labels = np.vstack(all_labels)

    avg_loss = total_loss / len(dataloader)
    metrics = compute_metrics(all_predictions, all_labels)

    return avg_loss, metrics

print("✓ Funciones de evaluación definidas")

In [ ]:
# Entrenamiento
print(f"\n🚀 Iniciando entrenamiento por {config.EPOCHS} épocas...")
print(f"   Dispositivo: {device}")
print(f"   Batch size: {config.TRAIN_BATCH_SIZE}")
print(f"   Acumulación: {config.ACCUMULATION_STEPS}")

best_f1 = 0
history = {'train_loss': [], 'val_loss': [], 'val_f1_micro': [], 'val_f1_macro': []}

for epoch in range(config.EPOCHS):
    print(f"\n📊 Época {epoch + 1}/{config.EPOCHS}")

    # ==== ENTRENAMIENTO ====
    model.train()
    train_loss = 0
    optimizer.zero_grad()

    progress_bar = tqdm(train_loader, desc=f"Entrenando época {epoch+1}")
    for step, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss = loss / config.ACCUMULATION_STEPS

        # Backward pass
        loss.backward()

        if (step + 1) % config.ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        train_loss += loss.item() * config.ACCUMULATION_STEPS
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_train_loss = train_loss / len(train_loader)
    history['train_loss'].append(avg_train_loss)

    # ==== VALIDACIÓN ====
    val_loss, val_metrics = evaluate(model, val_loader, criterion, device)
    history['val_loss'].append(val_loss)
    history['val_f1_micro'].append(val_metrics['f1_micro'])
    history['val_f1_macro'].append(val_metrics['f1_macro'])

    print(f"\n   Train Loss: {avg_train_loss:.4f}")
    print(f"   Val Loss:   {val_loss:.4f}")
    print(f"   Val F1 (micro): {val_metrics['f1_micro']:.4f}")
    print(f"   Val F1 (macro): {val_metrics['f1_macro']:.4f}")
    print(f"   Val Precision: {val_metrics['precision_micro']:.4f}")
    print(f"   Val Recall:    {val_metrics['recall_micro']:.4f}")

    # Guardar mejor modelo
    if val_metrics['f1_micro'] > best_f1:
        best_f1 = val_metrics['f1_micro']
        print(f"   💾 Nuevo mejor modelo! F1: {best_f1:.4f}")

        # Guardar modelo completo
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'f1_micro': best_f1,
            'config': config.__dict__
        }, MODEL_DIR / 'best_model.pt')

        # Guardar solo modelo para inferencia
        model.roberta.save_pretrained(MODEL_DIR)
        tokenizer.save_pretrained(MODEL_DIR)

print(f"\n✅ Entrenamiento completado!")
print(f"   Mejor F1 (micro): {best_f1:.4f}")

In [ ]:
# Instalar/importar dependencias RAG
try:
    import faiss
    from sentence_transformers import SentenceTransformer
except ImportError:
    print("Instalando dependencias RAG...")
    !pip install -q faiss-cpu sentence-transformers
    import faiss
    from sentence_transformers import SentenceTransformer

print("📚 Configurando sistema RAG...")

In [ ]:
class RAGExplainer:
    """Sistema RAG para explicar predicciones CIE-10"""

    def __init__(self, model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'):
        self.encoder = SentenceTransformer(model_name)
        self.index = None
        self.chunks = []
        self.code_descriptions = {}

    def load_cie10_manual(self, manual_path=None):
        """Cargar manual CIE-10 (simulado con descripciones)"""
        # En producción, cargar desde archivo real del manual
        # Por ahora, crear descripciones básicas desde los códigos
        print("📖 Cargando manual CIE-10...")

        # Cargar descripciones de códigos desde CODIESP
        for code in all_codes:
            # En producción: buscar en manual oficial
            self.code_descriptions[code] = {
                'code': code,
                'description': f'Descripción del código {code}',
                'category': code.split('.')[0] if '.' in code else code
            }

        # Crear chunks del manual
        for code, info in self.code_descriptions.items():
            chunk = f"Código {code}: {info['description']}"
            self.chunks.append({
                'code': code,
                'text': chunk,
                'metadata': info
            })

        print(f"✓ {len(self.chunks)} chunks cargados")

    def build_index(self):
        """Construir índice FAISS"""
        print("🔨 Construyendo índice vectorial...")

        # Extraer textos
        texts = [chunk['text'] for chunk in self.chunks]

        # Generar embeddings
        embeddings = self.encoder.encode(texts, show_progress_bar=True)

        # Crear índice FAISS
        dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dimension)
        self.index.add(embeddings.astype('float32'))

        print(f"✓ Índice construido ({dimension} dimensiones)")

    def search(self, query, top_k=5):
        """Buscar chunks relevantes"""
        query_embedding = self.encoder.encode([query])
        distances, indices = self.index.search(query_embedding.astype('float32'), top_k)

        results = []
        for idx, dist in zip(indices[0], distances[0]):
            results.append({
                'chunk': self.chunks[idx],
                'distance': float(dist),
                'score': 1 / (1 + dist)  # Convertir distancia a score
            })

        return results

    def explain_prediction(self, text, predicted_codes, top_k=3):
        """Generar explicación para predicción"""
        explanations = []

        for code in predicted_codes:
            # Buscar contexto relevante en el manual
            query = f"{text} {code}"
            results = self.search(query, top_k=top_k)

            # Construir explicación
            explanation = {
                'code': code,
                'description': self.code_descriptions.get(code, {}).get('description', 'N/A'),
                'evidence': [
                    {
                        'text': r['chunk']['text'],
                        'relevance': r['score']
                    }
                    for r in results[:top_k]
                ],
                'reasoning': f"El código {code} fue detectado en base al texto clínico."
            }
            explanations.append(explanation)

        return explanations

    def save(self, path):
        """Guardar índice"""
        os.makedirs(path, exist_ok=True)
        faiss.write_index(self.index, str(path / 'rag.index'))

        with open(path / 'chunks.json', 'w') as f:
            json.dump(self.chunks, f, indent=2, ensure_ascii=False)

        with open(path / 'code_descriptions.json', 'w') as f:
            json.dump(self.code_descriptions, f, indent=2, ensure_ascii=False)

        print(f"✓ RAG guardado en {path}")

# Inicializar sistema RAG
rag_explainer = RAGExplainer()
rag_explainer.load_cie10_manual()
rag_explainer.build_index()

# Guardar
rag_explainer.save(RAG_DIR)

print("\n✅ Sistema RAG configurado y guardado")

In [ ]:
# Función de inferencia
def predict_with_explanation(text, model, tokenizer, rag_explainer, threshold=0.5, top_k=5):
    """Predecir códigos con explicación RAG"""
    model.eval()

    # Tokenizar
    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=config.MAX_LENGTH,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # Predecir
    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs = torch.sigmoid(logits).cpu().numpy()[0]

    # Obtener top-k códigos predichos
    top_indices = np.argsort(probs)[::-1][:top_k]
    top_indices = [idx for idx in top_indices if probs[idx] >= threshold]

    predictions = []
    for idx in top_indices:
        code = id2code[idx]
        confidence = float(probs[idx])
        predictions.append({
            'code': code,
            'confidence': confidence
        })

    # Generar explicaciones con RAG
    predicted_codes = [p['code'] for p in predictions]
    explanations = rag_explainer.explain_prediction(text, predicted_codes)

    return {
        'text': text,
        'predictions': predictions,
        'explanations': explanations
    }

# Prueba con texto de ejemplo
test_text = """
Paciente de 65 años que acude por dolor torácico y disnea.
Antecedentes de hipertensión arterial esencial.
Diagnóstico: Angina de pecho estable.
"""

print("🧪 Prueba de inferencia:\n")
print(f"Texto: {test_text}\n")

result = predict_with_explanation(test_text, model, tokenizer, rag_explainer, threshold=0.3, top_k=3)

print("Predicciones:")
for pred in result['predictions']:
    print(f"  - {pred['code']} (confianza: {pred['confidence']:.3f})")

print("\nExplicaciones:")
for exp in result['explanations']:
    print(f"\n  Código: {exp['code']}")
    print(f"  Descripción: {exp['description']}")
    print(f"  Razonamiento: {exp['reasoning']}")
    print(f"  Evidencia:")
    for ev in exp['evidence'][:2]:
        print(f"    - {ev['text'][:100]}... (relevancia: {ev['relevance']:.3f})")

## 10. Prueba de Inferencia con Explicabilidad

Test del modelo entrenado con explicaciones RAG.

## 9. Sistema RAG para Explicabilidad

Implementación de Retrieval-Augmented Generation para justificar las predicciones usando el manual CIE-10.

## 8. Loop de Entrenamiento

Entrenamiento del modelo con validación y guardado de checkpoints.

## 7. Configuración del Entrenamiento

Optimizador, scheduler y función de pérdida para clasificación multilabel.

## 6. Modelo RoBERTa-BSC para Clasificación Multilabel

## 5. Carga del Tokenizer y Preparación de Datos

## 4. Dataset y DataLoader Personalizados

## 3. Carga y Exploración del Dataset CODIESP

Carga del corpus de informes clínicos españoles con sus códigos CIE-10.

## 2. Configuración de Hiperparámetros

Configuración optimizada para CPU (ajustable para GPU).

## 1. Configuración del Entorno

Instalación de dependencias necesarias para el entrenamiento.